<a href="https://colab.research.google.com/github/dhwlxor/My-Ropo/blob/main/CD_%ED%8C%8C%EC%9D%BC_%EB%B3%B5%EC%82%AC%EA%B8%B0_%EA%B2%80%EC%A6%9D_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile cd_verifier_windows.py

import os
import hashlib
import time
import datetime
import tkinter as tk
from tkinter import filedialog
import pyautogui
import pygetwindow as gw

def calculate_hash(file_path):
    """파일의 지문(SHA256)을 계산합니다."""
    sha256_hash = hashlib.sha256()
    try:
        with open(file_path, "rb") as f:
            for byte_block in iter(lambda: f.read(4096), b""):
                sha256_hash.update(byte_block)
        return sha256_hash.hexdigest()
    except:
        return None

def get_folder_path(prompt_title):
    """윈도우 폴더 선택창을 띄웁니다."""
    root = tk.Tk()
    root.withdraw()  # 빈 메인 창 숨기기
    root.attributes('-topmost', True)  # 선택창을 맨 앞으로
    path = filedialog.askdirectory(title=prompt_title)
    root.destroy()
    return path

def capture_result():
    """결과 화면을 자동으로 캡처합니다."""
    try:
        time.sleep(1)  # 화면이 다 그려질 때까지 대기

        # 현재 활성화된 창(검은색 콘솔) 찾기
        win = gw.getActiveWindow()

        if win:
            # 타임스탬프 파일명 생성
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"Result_{timestamp}.jpg"

            # 해당 창 영역만 캡처
            img = pyautogui.screenshot(region=(win.left, win.top, win.width, win.height))
            img.save(filename)
            print(f"\n📸 결과 캡처 완료: {filename}")
        else:
            print("\n⚠️ 캡처할 창을 찾지 못했습니다.")

    except Exception as e:
        print(f"\n❌ 캡처 중 오류 발생: {e}")

def verify_data(src_dir, tgt_dir):
    # 시스템 파일 제외 목록
    IGNORE = ['desktop.ini', 'thumbs.db', '.ds_store']

    print("\n" + "="*50)
    print("📀 CD-RW 데이터 무결성 검사 (Windows)")
    print("="*50)
    print(f"원본: {src_dir}")
    print(f"대상: {tgt_dir}")
    print("-" * 50)

    # 검사할 파일 목록 수집
    files_to_check = []
    for root, dirs, files in os.walk(src_dir):
        for file in files:
            if file.lower() in IGNORE:
                continue
            # 상대 경로 계산
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, src_dir)
            files_to_check.append(rel_path)

    total = len(files_to_check)
    success = 0
    failed_list = []

    # 하나씩 비교 시작
    for i, rel_path in enumerate(files_to_check, 1):
        src_path = os.path.join(src_dir, rel_path)
        tgt_path = os.path.join(tgt_dir, rel_path)

        print(f"[{i}/{total}] {rel_path} ... ", end="", flush=True)

        # 1. 파일 존재 확인
        if not os.path.exists(tgt_path):
            print("❌ 미복사")
            failed_list.append((rel_path, "미복사"))
            continue

        # 2. 파일 크기 확인
        if os.path.getsize(src_path) != os.path.getsize(tgt_path):
            print("❌ 크기 다름")
            failed_list.append((rel_path, "크기 불일치"))
            continue

        # 3. 정밀 해시 확인
        if calculate_hash(src_path) == calculate_hash(tgt_path):
            print("✅ 일치")
            success += 1
        else:
            print("❌ 데이터 손상")
            failed_list.append((rel_path, "데이터 변조"))

    # 최종 결과 보고
    print("\n" + "="*50)
    print(f"📊 최종 결과: {success} / {total} 성공")

    if failed_list:
        print(f"⚠️ 실패: {len(failed_list)}건")
        for f, r in failed_list:
            print(f" - {f} ({r})")
    else:
        print("✨ 완벽하게 복사되었습니다!")
    print("="*50)

    # 캡처 실행
    capture_result()

    print("\n3초 뒤 종료됩니다...")
    time.sleep(3)

if __name__ == "__main__":
    # 프로그램 시작 시 폴더 선택창 띄우기
    print("폴더 선택창을 확인해주세요...")
    source = get_folder_path("1. 원본 데이터 폴더 선택")
    target = get_folder_path("2. 검사할 CD/대상 폴더 선택")

    if source and target:
        verify_data(source, target)
    else:
        print("❌ 경로가 선택되지 않았습니다.")
        time.sleep(2)

Overwriting cd_verifier_windows.py
